# Chip bag weights: the 1 g rule from a summary sheet

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**Hypothesis Testing for the Variance** · a worked example from the StatNex teaching guide

**The story.** A snack factory fills 150 g bags of chips on two packing lines. Its quality rule says the standard deviation of bag weight must not exceed 1 g. The summary sheet reports only the sample size (14 bags per line) and the standard deviation for each line.

**The question.** Can the quality team run the variance check from the summary sheet?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `02_one_sample/one_sample_variance_test__summary.csv` (2 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
Variable,Line A,Line B
Sample size,14.0,14.0
Standard deviation,0.6495,1.9652
""")

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.6g}".format)


def stars(p):
    """* for P < 0.05, ** for P < 0.01."""
    return "**" if p < 0.01 else "*" if p < 0.05 else ""

## The data

In [ ]:
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
variables = ["Line A", "Line B"]
# The population standard deviation under the null hypothesis.
sigma0 = 1.0

## Chi-square test for the variance, one column at a time

In [ ]:
# chi-square = (n - 1) s² / σ0²: if the true SD were σ0, it would land near n - 1, its
# degrees of freedom. Far above means more spread than σ0, far below means less.
# SciPy has no one-line test for this, so the chi-square distribution is used directly.
# The test assumes normal data and, unlike the t test, does not become safe as n
# grows: heavy tails make it reject far more often than it should.
results = {}
for name in variables:
    # Summary data: each column holds the sample size and the standard deviation,
    # top to bottom.
    size, sd = pd.to_numeric(data[name], errors="coerce").dropna().to_numpy(float)[:2]
    n = int(round(size))
    df = n - 1
    variance = sd**2
    ratio = variance / sigma0**2
    chi2 = df * ratio
    # The one-sided P value is the smaller tail -- the side the observed variance lies
    # on -- and the two-sided P value is twice that, capped at 1 (Stata's sdtest does
    # the same). Use the one-sided P only if its direction was fixed in advance.
    p_one = min(stats.chi2.cdf(chi2, df), stats.chi2.sf(chi2, df))
    p_two = min(1.0, 2 * p_one)
    results[name] = {"n": n, "SD": sd, "variance": variance, "sigma0^2": sigma0**2,
                     "ratio": ratio, "chi-square": chi2, "df": df,
                     "P two-sided": p_two, "P one-sided": p_one}

table = pd.DataFrame(results)
print(table)
print()
print(table.loc[["P two-sided", "P one-sided"]].map(stars))

## What the result says

This run uses only the summary sheet: 14 bags per line and each line's standard deviation, the typical distance of a bag from the average. These are 0.6495 g for Line A and 1.9652 g for Line B.

The test compares each variance (the standard deviation squared) with the rule's variance of 1. For Line B the chi-square statistic is 50.2 and the two-sided P value is 0.000005. A P value is the chance of a result at least this extreme if the true standard deviation were exactly 1 g. For Line A, chi-square is 5.48 and the one-sided P value is 0.037.

The verdicts match the raw weights: Line B breaks the rule and Line A stays below it. But the sheet cannot say why Line B fails, or whether the test can be trusted on these bags.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.